In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from src.features import BASIC_APPLICATION_FEATURES, create_features
from src.preprocessing import build_preprocessor
from src.evaluation import compare_model_cv

In [2]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    StratifiedKFold,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

In [3]:
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

train_df = pd.read_csv(
    RAW_DATA_DIR / "application_train.csv"
)

train_df.shape

(307511, 122)

In [4]:
selected_features = BASIC_APPLICATION_FEATURES


In [5]:
X = train_df[selected_features].copy()
y = train_df["TARGET"].copy()

X = create_features(X)

In [6]:
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

In [7]:
numeric_columns = X_train.select_dtypes(
    include="number"
).columns.tolist()

categorical_columns = X_train.select_dtypes(
    exclude="number"
).columns.tolist()

print("Numeric columns:", len(numeric_columns))
print("Categorical columns:", len(categorical_columns))

Numeric columns: 26
Categorical columns: 6


In [8]:
def build_model_pipeline(model):
    """
    Build a fresh preprocessing and model Pipeline.
    """

    scale_numeric = isinstance(model, LogisticRegression)

    preprocessor = build_preprocessor(
        numeric_columns=numeric_columns,
        categorical_columns=categorical_columns,
        scale_numeric=scale_numeric,
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )


In [9]:
models = {
    "Dummy Classifier": DummyClassifier(
        strategy="prior"
    ),

    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

In [10]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [11]:
results = []

for model_name, model in models.items():
    print(f"Evaluating: {model_name}")

    model_pipeline = build_model_pipeline(model)

    result = compare_model_cv(
        name=model_name,
        model_pipeline=model_pipeline,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring="roc_auc",
    )

    results.append(result)

Evaluating: Dummy Classifier


Evaluating: Logistic Regression


Evaluating: Decision Tree


Evaluating: Random Forest


In [12]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="mean_cv_roc_auc",
    ascending=False,
).reset_index(drop=True)

display(
    results_df.style.format({
        "mean_cv_roc_auc": "{:.4f}",
        "std_cv_roc_auc": "{:.4f}",
        "min_cv_roc_auc": "{:.4f}",
        "max_cv_roc_auc": "{:.4f}",
    })
)


,model,mean_cv_roc_auc,std_cv_roc_auc,min_cv_roc_auc,max_cv_roc_auc
0,Random Forest,0.7443,0.0020,0.7420,0.7472
1,Logistic Regression,0.7417,0.0023,0.7394,0.7461
2,Decision Tree,0.7242,0.0017,0.7212,0.7261
3,Dummy Classifier,0.5000,0.0000,0.5000,0.5000


Random Forest performed the best
